# indy_mech_extension — probe training (CPU runtime)
Pulls the teacher-forced features from the private HF dataset repo, trains mass-mean / LDA / logistic probes on the
528 phase-17 rollouts (leave-one-trigger-out), then scores phase 19's 8,000 rollouts with the best cells.
Scripts `train_probes.py`, `apply_probes_p19.py`, `labels.json`, `p19_labels.json` are uploaded from the local repo.

In [ ]:
# === CELL 1 — pull features from HF ==========================================================
import os, time, json, shutil, hashlib
from google.colab import userdata
from huggingface_hub import hf_hub_download, HfApi
tok = userdata.get("HF_TOKEN"); api = HfApi(token=tok)
REPO = f"{api.whoami()['name']}/indy-mech-extension-qwen3-8b-persona-probes"
os.makedirs("/content/work/phase19", exist_ok=True)
t0 = time.time()
for f in ["features_qwen_wide.npz", "features_meta.json"]:
    shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok), f"/content/work/{f}")
    print("  got", f, f"[{time.time()-t0:.0f}s]")
for f in ["p19_meta.json", "p19_ids.json", "p19_md5.json", "p19_A.npy", "p19_C.npy", "p19_B.npy", "p19_D.npy"]:
    shutil.copy(hf_hub_download(REPO, "phase19/" + f, repo_type="dataset", token=tok), f"/content/work/phase19/{f}")
    print("  got phase19/" + f, f"[{time.time()-t0:.0f}s]")
md5 = json.load(open("/content/work/phase19/p19_md5.json"))
for f, h in md5.items():
    p = f"/content/work/phase19/{f}"
    if os.path.exists(p): assert hashlib.md5(open(p, "rb").read()).hexdigest() == h, f"md5 mismatch {f}"
print("md5 OK; disk:", shutil.disk_usage("/content").free // 2**30, "GB free")
import sklearn, numpy; print("sklearn", sklearn.__version__, "numpy", numpy.__version__, "cpus", os.cpu_count())


In [ ]:
# === CELL 2 — mass-mean grid: 19 layers x 10 slots, both targets, both tiers, with/without nulls ==
import subprocess, sys
r = subprocess.run([sys.executable, "/content/work/train_probes.py", "--probes", "massmean", "--jobs", str(os.cpu_count()),
                    "--features", "/content/work/features_qwen_wide.npz", "--meta", "/content/work/features_meta.json",
                    "--labels", "/content/work/labels.json", "--out", "/content/work/results_massmean.json"],
                   capture_output=True, text=True); print(r.stdout[-6000:]); print(r.stderr[-3000:])


In [ ]:
# === CELL 3 — LDA + logistic at a reduced layer set (every 4th layer + final), all slots ============
r = subprocess.run([sys.executable, "/content/work/train_probes.py", "--probes", "massmean,lda,logistic", "--fixedC", "0.1",
                    "--layers", "0,4,8,12,16,20,24,28,32,36", "--n_perm", "0", "--jobs", str(os.cpu_count()),
                    "--features", "/content/work/features_qwen_wide.npz", "--meta", "/content/work/features_meta.json",
                    "--labels", "/content/work/labels.json", "--out", "/content/work/results_full.json"],
                   capture_output=True, text=True); print(r.stdout[-6000:]); print(r.stderr[-3000:])


In [ ]:
# === CELL 4 — score phase 19 with the best cells ==============================================
r = subprocess.run([sys.executable, "/content/work/apply_probes_p19.py", "--features", "/content/work/features_qwen_wide.npz",
                    "--meta", "/content/work/features_meta.json", "--labels", "/content/work/labels.json",
                    "--results", "/content/work/results_massmean.json", "--p19dir", "/content/work/phase19",
                    "--p19labels", "/content/work/p19_labels.json", "--out", "/content/work/p19_probe_scores.json"],
                   capture_output=True, text=True); print(r.stdout[-6000:]); print(r.stderr[-3000:])


In [ ]:
# === CELL 5 — push results back to the HF repo ===============================================
for f in ["results_massmean.json", "results_full.json", "p19_probe_scores.json"]:
    p = f"/content/work/{f}"
    if os.path.exists(p):
        api.upload_file(path_or_fileobj=p, path_in_repo="results/" + f, repo_id=REPO, repo_type="dataset"); print("  uploaded", f, os.path.getsize(p)//1000, "KB")
